1. Load ENV Variable 

In [6]:
import os 
from getpass import getpass
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API Key: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter your TAVILY API Key: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter your PINECONE API Key: ")

print("GROQ API Key:",bool(os.getenv("GROQ_API_KEY")))
print("TAVILY API Key:",bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE API Key:",bool(os.getenv("PINECONE_API_KEY")))




GROQ API Key: True
TAVILY API Key: True
PINECONE API Key: True


2. LOAD DOCUMENTS 

In [7]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    requests_kwargs={
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
            
        }
    },
)

raw_docs = loader.load()

print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500].encode("utf-8", errors="replace").decode("utf-8"))


C:\Users\Aman\AppData\Local\Temp\ipykernel_41384\1504425881.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\Aman\Desktop\RAG\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional AP

3. SPILTING THE DOCUMENT INTO CHUNKS 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spiltter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)

chunks = spiltter.split_documents(raw_docs)

print("Split into chunks:", len(chunks))
print("Preview of first chunk:\n")
print(chunks[0].page_content[:1000])


Split into chunks: 28
Preview of first chunk:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangCh

4. LOCAL EMBEDDINGS

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)
sample_embedding = embeddings.embed_query("What is Agentic RAG?")
print("Embedding dimensions:", len(sample_embedding))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10264.28it/s]


Embedding dimensions: 384


5. CREATING PINECONE VECTOR DATABASE 

In [10]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create the index only if it does not already exist.
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,       # all-MiniLM-L6-v2 embedding dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Wait until Pinecone reports the new index as ready.
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

print("Pinecone index ready:", INDEX_NAME)

# Upload the document chunks and create the LangChain vector store.
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Pinecone vector database and retriever are ready.")


Pinecone index ready: industry-agentic-rag-kb
Pinecone vector database and retriever are ready.


In [11]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Load existing Pinecone index
index = pc.Index(INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Existing Pinecone index loaded:", INDEX_NAME)


Existing Pinecone index loaded: industry-agentic-rag-kb


6. TEST KB


In [13]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):
    print(f"\n--- KB RESULT {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])



--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL as

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: 

7. initialize Groq LLM

In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

print (llm.invoke("What is Agentic RAG?").content)


## Agentic Retrieval‑Augmented Generation (Agentic RAG)

**Agentic RAG** is a recent design pattern for large‑language‑model (LLM) agents that combines the classic *Retrieval‑Augmented Generation* (RAG) pipeline with *agentic* (autonomous, goal‑driven) capabilities.  
In short: it is a RAG system that can *plan*, *act*, *evaluate*, and *refine* its own queries and responses, rather than simply fetching a static set of documents and generating a single answer.

---

### 1. Quick recap: What is RAG?

| Component | What it does | Typical implementation |
|-----------|--------------|------------------------|
| **Retriever** | Finds relevant documents from a knowledge base (KB) given a user query. | BM25, DPR, FAISS, ElasticSearch, etc. |
| **Generator** | Generates a final answer conditioned on the query **and** the retrieved passages. | GPT‑4, Llama‑2‑Chat, etc. |
| **Fusion** | Combines retrieved passages with the query before feeding to the generator. | Concatenation, cross‑attention, r

8. Init Tavily web search

In [15]:

from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=False,
)

print("Tavily search tool ready.")


Tavily search tool ready.


9. define structure decisions


In [16]:
from typing import List, Literal,Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document

class RouteDecision(BaseModel):
    route:Literal["kb","direct"]=Field(
        description="use kb for questions needing agentic RAG docs ;direct for greeting/simple chat."
    )
class EvidenceGrade(BaseModel):
    grade:Literal["good","weak"]=Field(
        description="good means evidence can answer the question ; weak means not enough evidence to answer the question"
    )

class AgentState(TypedDict):
    question:str
    current_query:str
    kb_docs:List[Document]
    web_results:str
    kb_grade:str
    web_grade:str
    answer:str
    source_used:str
    retry_count:int


10. Node 1 - Route the Questions

In [17]:
router_llm = llm.with_structured_output(RouteDecision, method="json_mode")

def route_question(state:AgentState):
    question=state["question"]
    decision=router_llm.invoke(f'''
You are a router for an Agentic RAG assistant.

Route to "kb" if the user asks about:
- Agentic RAG
- LangGraph Agentic RAG workflow
- retrieval grading
- query rewriting
- RAG architecture
- retriever tools
- web fallback in RAG

Route to "direct" only for greetings, thanks, or very simple conversation.

Question:
{question}

Return your response as valid JSON.
Example:
{{"route": "kb"}}
''')
    print ("[ROUTER DECISION]",decision.route)
    return{
        "current_query":question,
        "source_used":decision.route,
    }

def route_after_router(state:AgentState)->Literal["retrieve_kb","direct_answer"]:
    if state["source_used"]=="kb":
        return "retrieve_kb"
    else:
        return "direct_answer"

